# Transformers playground

Run the two setup cells, then jump to any part.

1. **The GPU**
2. **Hugging Face**
3. **Tokens**
4. **ModernBERT**, a word in context
5. **One vector per passage**, and search
6. **CLIP**, inside and out

Free Colab runs all of it.

> **File → Save a copy in Drive** first. Colab wipes its disk when the runtime ends.

In [ ]:
%pip install -q -U transformers sentence-transformers

In [ ]:
import os, re, time
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()                 # quieten the load reports

print("torch", torch.__version__)

## 1 · The GPU

A GPU does thousands of multiplications at once, which is all a transformer does.

Colab: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
if torch.cuda.is_available():
    DEVICE = "cuda"
    print("GPU:", torch.cuda.get_device_name(0),
          round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
elif torch.backends.mps.is_available():          # Apple silicon, running locally
    DEVICE = "mps"
    print("Apple GPU")
else:
    DEVICE = "cpu"
    print("No GPU. Everything still works, slower.")
print("device:", DEVICE)

In [ ]:
# One big matrix multiply, timed on each device you have.
def time_matmul(device, n=2000, reps=5):
    a = torch.randn(n, n, device=device)
    b = torch.randn(n, n, device=device)
    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(reps):
        a @ b
    if device == "cuda":
        torch.cuda.synchronize()
    return (time.time() - t0) / reps

cpu = time_matmul("cpu")
print(f"cpu   {cpu*1000:7.0f} ms")
if DEVICE != "cpu":
    gpu = time_matmul(DEVICE)
    print(f"{DEVICE:5s} {gpu*1000:7.0f} ms   ({cpu/gpu:.0f}x faster)")

## 2 · Hugging Face

Models live at [huggingface.co](https://huggingface.co/models), named `account/model`. Read the
model card. `pipeline` downloads the model, the tokenizer and the code around both.

In [ ]:
from transformers import pipeline

sentiment = pipeline("sentiment-analysis", device=DEVICE)
for line in ["The sequel was two hours I will never get back.",
             "I have not stopped thinking about the last twenty minutes.",
             "It was, I suppose, a film."]:
    out = sentiment(line)[0]
    print(f"{out['label']:8s} {out['score']:.2f}  {line}")

print("\nyou just used:", sentiment.model.config._name_or_path)

You did not pick that model, and it was trained on product reviews. Do you agree with the
third line?

In [ ]:
# Where the downloads go. Delete this folder to reclaim the disk.
from pathlib import Path
cache = Path(os.environ.get("HF_HOME", Path.home() / ".cache/huggingface"))
print(cache)
if cache.exists():
    size = sum(f.stat().st_size for f in cache.rglob("*") if f.is_file())
    print(f"{size/1e9:.2f} GB cached")

## 3 · Tokens

A model sees **tokens**, not letters: common words whole, rare ones in pieces.

In [ ]:
from transformers import AutoTokenizer

MODEL = "answerdotai/ModernBERT-base"
tok = AutoTokenizer.from_pretrained(MODEL)

for line in ["I ate the whole pizza.",
             "antidisestablishmentarianism",
             "Nosferatu, Whitby, Bistritz",
             "https://example.com/page?id=42"]:
    pieces = [p.replace("\u0120", " ").strip() for p in tok.tokenize(line)]
    print(f"{len(pieces):3d} | " + " / ".join(pieces))

In [ ]:
# Tokens are numbers. This is the actual input to every model below.
enc = tok("A museum keeps its collection.")
print(enc["input_ids"])
print(tok.convert_ids_to_tokens(enc["input_ids"]))

## 4 · ModernBERT

**ModernBERT** (2024), a rebuilt BERT: 22 layers, 8,192 tokens of context. An *encoder*, so it
reads rather than writes, returning one vector per token.

It was trained by filling in blanks.

In [ ]:
fill = pipeline("fill-mask", model=MODEL, device=DEVICE)

for s in ["I put the milk in the [MASK].",
          "The best thing about Chicago is the [MASK].",
          "Mary Shelley wrote [MASK] in 1818.",
          "My code failed because I forgot a [MASK]."]:
    print(f"{s:46s} {[g['token_str'].strip() for g in fill(s, top_k=4)]}")

The Shelley line is grammatical, not true.

### The same word, four sentences

In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained(MODEL).to(DEVICE).eval()

def token_span(sentence, word):
    """Where `word` sits in the token list."""
    enc = tok(sentence, return_tensors="pt")
    ids = enc["input_ids"][0].tolist()
    want = tok(" " + word, add_special_tokens=False)["input_ids"]
    at = [i for i in range(len(ids) - len(want) + 1) if ids[i:i+len(want)] == want]
    if not at:
        raise ValueError(f"'{word}' not found as a whole token in: {sentence}")
    return enc, at[0], len(want)

def word_vector(sentence, word):
    enc, i, n = token_span(sentence, word)
    with torch.no_grad():
        states = model(**enc.to(DEVICE)).last_hidden_state[0]
    v = states[i:i+n].mean(0)
    return (v / v.norm()).cpu().numpy()

sentences = ["A bat flew out of the cave at dusk.",
             "The bat hung upside down all winter.",
             "She swung the bat and missed.",
             "He gripped the bat with both hands."]
V = np.stack([word_vector(s, "bat") for s in sentences])

print("     " + "".join(f"   s{i+1}" for i in range(4)))
for i, row in enumerate(V @ V.T):
    print(f"  s{i+1} " + "".join(f" {x:+.2f}" for x in row) + "   " + sentences[i])

Animals together, baseball bats together, the pairs apart. GloVe gives *bat* one vector. This
gives it one per sentence.

Every number is high, because raw encoder vectors all point roughly the same way. Read the
blocks, not the values. Part 5 fixes that.

## 5 · One vector per passage

To compare passages you need one vector each. Three ways:

- **CLS**, the first token.
- **Mean**, the average of the token vectors.
- **A trained sentence model**, fine-tuned so its vectors are comparable.

Test: 240 passages from *Dracula* and *Frankenstein*. Nearest neighbour, same book?

In [ ]:
def passages(path, book, target=120):
    raw = open(path, encoding="utf-8", errors="ignore").read()
    a, b = raw.find("*** START OF"), raw.find("*** END OF")
    body = raw[raw.find("\n", a) + 1:b] if a > 0 else raw
    paras = [" ".join(p.split()) for p in re.split(r"\n\s*\n", body)]
    paras = [p for p in paras if len(p.split()) >= 60]
    step = max(1, len(paras) // target)
    return [(book, p[:900]) for p in paras[::step]][:target]

for base in ("data/texts", "notebooks/data/texts", "../notebooks/data/texts"):
    if os.path.isdir(base):
        break
else:
    raise FileNotFoundError("clone the repo first, or point `base` at your own texts")

rows = (passages(f"{base}/dracula.txt", "Dracula")
        + passages(f"{base}/frankenstein.txt", "Frankenstein"))
books = np.array([b for b, _ in rows])
texts = [t for _, t in rows]
print(len(rows), "passages")
print(texts[3][:120], "...")

In [ ]:
def pool(texts, how, batch=16):
    out = []
    for i in range(0, len(texts), batch):
        enc = tok(texts[i:i+batch], return_tensors="pt", padding=True,
                  truncation=True, max_length=256).to(DEVICE)
        with torch.no_grad():
            h = model(**enc).last_hidden_state
        m = enc["attention_mask"].unsqueeze(-1)
        v = h[:, 0] if how == "cls" else (h * m).sum(1) / m.sum(1)
        out.append(v.float().cpu().numpy())
    V = np.vstack(out)
    return V / np.linalg.norm(V, axis=1, keepdims=True)

from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)

ways = {
    "CLS, untrained":  pool(texts, "cls"),
    "mean, untrained": pool(texts, "mean"),
    "MiniLM, trained": embedder.encode(texts, normalize_embeddings=True, batch_size=32),
}
for name, V in ways.items():
    S = V @ V.T
    np.fill_diagonal(S, -9)
    right = (books[S.argmax(1)] == books).mean()
    off = S[np.triu_indices(len(V), 1)]
    print(f"{name:17s} neighbour from same book {right:.0%}   "
          f"cosines {off.min():+.2f} to {off.max():+.2f}")

Read the range, not the accuracy. Untrained, unrelated paragraphs score 0.96 and so do
near-identical ones. Trained, the number means something.

In [ ]:
S = ways["MiniLM, trained"]

def search(question, k=3):
    scores = S @ embedder.encode([question], normalize_embeddings=True)[0]
    for i in np.argsort(-scores)[:k]:
        print(f"  {scores[i]:.2f} {books[i]:13s} {' '.join(texts[i].split()[:16])}...")

for q in ["someone climbs down a wall",
          "a creature asks to be less alone",
          "the weather turns and the ship is in danger",
          "reading a letter from home"]:
    print("\n" + q)
    search(q)

None of those words appear in the passages.

In [ ]:
search("write your own question here", k=3)

In [ ]:
from sklearn.decomposition import PCA

pts = PCA(n_components=2).fit_transform(S)
plt.figure(figsize=(7, 5))
for book, colour in [("Dracula", "#A34526"), ("Frankenstein", "#2E6E8E")]:
    m = books == book
    plt.scatter(pts[m, 0], pts[m, 1], s=16, color=colour, alpha=0.75, label=book)
plt.xticks([]); plt.yticks([])
plt.legend(frameon=False)
plt.title("240 passages, arranged by meaning", loc="left")
plt.tight_layout(); plt.show()

Nobody told it which book. The overlap in the middle is correct.

## 6 · CLIP

400 million picture–caption pairs: pull each picture towards its own caption, push it from
everyone else's. Pictures and sentences end up in one space.

The 18 Met paintings below have no tags, only titles, which the model never sees. All 18 are
portraits, so ask portrait questions.

In [ ]:
import csv

for mbase in ("data/week01", "notebooks/data/week01", "../notebooks/data/week01"):
    if os.path.isdir(os.path.join(mbase, "met")):
        break
else:
    raise FileNotFoundError("clone the repo first")

rows = list(csv.DictReader(open(f"{mbase}/met_manifest.csv", encoding="utf-8")))
paths = [os.path.join(mbase, r["file"]) for r in rows]
titles = [r["title"] for r in rows]

clip = SentenceTransformer("clip-ViT-B-32", device=DEVICE)
pictures = clip.encode([Image.open(p) for p in paths], normalize_embeddings=True)
print(len(paths), "paintings,", pictures.shape[1], "numbers each")

In [ ]:
def look_for(phrase, k=3):
    scores = pictures @ clip.encode([phrase], normalize_embeddings=True)[0]
    best = np.argsort(-scores)[:k]
    fig, axes = plt.subplots(1, k, figsize=(3.4 * k, 3.4))
    for ax, i in zip(axes, best):
        ax.imshow(Image.open(paths[i]))
        ax.set_title(f"{scores[i]:.2f}  {titles[i][:28]}", loc="left", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f'"{phrase}"', x=0.02, ha="left", fontsize=12)
    plt.tight_layout(); plt.show()

look_for("a man in a straw hat")
look_for("a woman in a white headdress")
look_for("an artist at work")

Right answer every time, at 0.25 to 0.32. CLIP's scores sit in a narrow band; only the ranking
matters.

### Labels with no training

Labels as sentences, nearest one wins. No examples, no fitting. Edit the list and rerun.

In [ ]:
LABELS = ["a painting of a man", "a painting of a woman", "a landscape",
          "a religious painting", "a still life"]

scores = pictures @ clip.encode(LABELS, normalize_embeddings=True).T
for i, t in enumerate(titles):
    j = int(np.argmax(scores[i]))
    print(f"{LABELS[j]:26s} {scores[i, j]:.2f}   {t[:44]}")

Bronzino's *Portrait of a Young Man* comes out as woman, at 0.32, the same score as the ones it
gets right. The score is confidence, not correctness.

### What the picture looks like to it

A **Vision Transformer** cuts the image into a 7×7 grid of 32-pixel patches and treats each as a
token. Patches are to an image what tokens are to a sentence.

In [ ]:
from transformers import CLIPModel, CLIPProcessor

CLIP_ID = "openai/clip-vit-base-patch32"
vit = CLIPModel.from_pretrained(CLIP_ID, attn_implementation="eager").to(DEVICE).eval()
proc = CLIPProcessor.from_pretrained(CLIP_ID)

PICK = 0                                          # change this
img = Image.open(paths[PICK]).convert("RGB")
px = proc(images=img, return_tensors="pt")["pixel_values"]
square = np.array(img.resize((224, 224)))

fig, (a, b) = plt.subplots(1, 2, figsize=(9, 4.6))
a.imshow(square); a.set_title("what you see", loc="left", fontsize=10)
b.imshow(square)
for g in range(0, 225, 32):
    b.axvline(g, color="w", lw=1.2); b.axhline(g, color="w", lw=1.2)
b.set_title("what it sees: 49 patches", loc="left", fontsize=10)
for ax in (a, b):
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
print(px.shape, "->", 224 // 32, "x", 224 // 32, "patches + 1 CLS token")

In [ ]:
# Every patch on its own. This is the whole input, in order.
patches = square.reshape(7, 32, 7, 32, 3).transpose(0, 2, 1, 3, 4)
fig, axes = plt.subplots(7, 7, figsize=(5.6, 5.6))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(patches[i // 7, i % 7]); ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("49 tokens", x=0.02, ha="left", fontsize=12)
plt.tight_layout(); plt.show()

### Where it looks

The CLS token becomes the image's vector. Its attention row shows what fed into it.

In [ ]:
with torch.no_grad():
    att = vit.vision_model(px.to(DEVICE), output_attentions=True).attentions

def cls_map(layer):
    a = att[layer][0].mean(0)[0, 1:]              # average the heads, CLS row, drop CLS itself
    return a.reshape(7, 7).float().cpu().numpy()

# Dim each patch by how little the CLS token looked at it.
fig, axes = plt.subplots(1, 4, figsize=(12, 3.4))
axes[0].imshow(square / 255); axes[0].set_title("image", loc="left", fontsize=10)
for ax, layer in zip(axes[1:], (0, 5, 11)):
    m = cls_map(layer)
    lit = np.kron(m / m.max(), np.ones((32, 32)))[..., None]
    ax.imshow(square / 255 * (0.15 + 0.85 * lit))
    ax.set_title(f"layer {layer}", loc="left", fontsize=10)
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("where the CLS token looked", x=0.02, ha="left", fontsize=12)
plt.tight_layout(); plt.show()

for layer in (0, 5, 11):
    m = cls_map(layer)
    print(f"layer {layer:2d}  busiest patch gets {m.max()/m.mean():.1f}x the average")

Flat at layer 0. By layer 11 a few patches carry several times the average, mostly the face and
the hat.

Attention shows where information came from, not why: a hypothesis, not evidence. See Jain and
Wallace, *Attention is not Explanation* (2019).

## Next

- **Your own corpus.** Part 5 takes any list of strings.
- **A bigger embedder.** `all-mpnet-base-v2`, or the
  [MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard).
- **Fine-tuning.** `cool-methods/finetune_modernbert.ipynb`, once an off-the-shelf model has
  failed you.
- **Other pipelines.** `"zero-shot-classification"`, `"ner"`, `"summarization"`,
  `"image-classification"`.

Three things to keep saying:

1. Name the model and version. "An AI said" is not a method.
2. A model trained on the open web knows the open web, not your corpus.
3. A confident score is not a correct answer.